In [2]:
import pandas as pd
df = pd.read_csv("/kaggle/heart.csv")
df.head()


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0


In [3]:
X = df.drop(columns=["target"])
y = df["target"]

In [4]:
for col in X.columns:
    if X[col].dtype in ["int64", "float64"]:
        X[col] = X[col].fillna(X[col].median())
    else:
        X[col] = X[col].fillna(X[col].mode()[0])

print("Missing values:", X.isnull().sum().sum())

Missing values: 0


In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape)


(820, 13) (205, 13)


In [6]:
import torch

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

print(X_train_tensor.shape, y_train_tensor.shape)


torch.Size([820, 13]) torch.Size([820, 1])


In [7]:
import torch.nn as nn

class HeartDiseaseNN(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

model = HeartDiseaseNN(X_train_tensor.shape[1])
print(model)


HeartDiseaseNN(
  (model): Sequential(
    (0): Linear(in_features=13, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=16, bias=True)
    (3): ReLU()
    (4): Linear(in_features=16, out_features=1, bias=True)
    (5): Sigmoid()
  )
)


In [8]:
import torch.optim as optim

criterion = nn.BCELoss()           # checks how wrong prediction is Loss = −[ y log(p) + (1 − y) log(1 − p) ]

optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 100

for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")


Epoch 0, Loss: 0.7021
Epoch 10, Loss: 0.6781
Epoch 20, Loss: 0.6519
Epoch 30, Loss: 0.6183
Epoch 40, Loss: 0.5761
Epoch 50, Loss: 0.5270
Epoch 60, Loss: 0.4765
Epoch 70, Loss: 0.4299
Epoch 80, Loss: 0.3911
Epoch 90, Loss: 0.3612


In [9]:
model.eval()

with torch.no_grad():
    probs = model(X_test_tensor)
    predictions = (probs >= 0.5).float()

accuracy = (predictions == y_test_tensor).sum() / y_test_tensor.size(0)
print("Neural Model Accuracy:", accuracy.item())

print("Sample probabilities:", probs[:5].squeeze())


Neural Model Accuracy: 0.8048780560493469
Sample probabilities: tensor([0.9592, 0.9623, 0.1063, 0.9621, 0.2299])


In [10]:
def medical_rules(row, neural_prob):
    reasons = []

    # Rule 1: Very risky chest pain
    if row["cp"] in [2, 3]:
        reasons.append("High risk chest pain")

    # Rule 2: Low max heart rate
    if row["thalach"] < 100:
        reasons.append("Low maximum heart rate")

    # Rule 3: Exercise-induced angina
    if row["exang"] == 1:
        reasons.append("Exercise induced angina")

    # Rule 4: Strong neural confidence
    if neural_prob > 0.7:
        reasons.append("High neural risk score")

    # Final decision
    decision = 1 if len(reasons) > 0 else 0

    return decision, reasons


In [11]:
X_test_df = pd.DataFrame(X_test, columns=X.columns)

final_decisions = []

for i in range(len(X_test_df)):
    prob = probs[i].item()
    row = X_test_df.iloc[i]

    decision, reason = medical_rules(row, prob)

    final_decisions.append({
        "Neural_Probability": prob,
        "Final_Decision": decision,
        "Reasons": reason
    })

pd.DataFrame(final_decisions).head()


,Neural_Probability,Final_Decision,Reasons
0,0.959154,1,"[Low maximum heart rate, High neural risk score]"
1,0.962323,1,"[Low maximum heart rate, High neural risk score]"
2,0.106283,1,[Low maximum heart rate]
3,0.962105,1,"[Low maximum heart rate, High neural risk score]"
4,0.229927,1,[Low maximum heart rate]


**Neuro-symbolic AI combines neural networks and symbolic reasoning to create systems that can learn from data while applying logical reasoning, aiming for more robust and interpretable artificial intelligence.**


Medical Data

   ↓

Neural Network (Learning)

   ↓

Probability

   ↓

Medical Rules (Logic)

   ↓
   
Decision + Explanation


